## 内置工具

https://python.langchain.ac.cn/docs/integrations/tools/



In [1]:
!pip install -qU duckduckgo-search langchain-community ddgs

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-deepseek 1.0.1 requires langchain-openai<2.0.0,>=1.0.0, but you have langchain-openai 0.2.1 which is incompatible.
langchain 0.3.0 requires langchain-core<0.4.0,>=0.3.0, but you have langchain-core 1.2.7 which is incompatible.
langchain 0.3.0 requires langchain-text-splitters<0.4.0,>=0.3.0, but you have langchain-text-splitters 1.1.0 which is incompatible.
langchain 0.3.0 requires langsmith<0.2.0,>=0.1.17, but you have langsmith 0.6.4 which is incompatible.
langchain-openai 0.2.1 requires langchain-core<0.4,>=0.3, but you have langchain-core 1.2.7 which is incompatible.


In [2]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("苹果公司的创始人 ?")



'May 5, 2025 · 史蒂芬·保罗·乔布斯（英语：Steven Paul Jobs，1955年2月24日－2011年10月5日），通称史蒂夫·乔布斯（英语：Steve Jobs），是一名美国企业家、营销家搭发明家，苹果公司个联合创始人之 ... Dec 17, 2025 · 史蒂夫·乔布斯（Steve Jobs，1955年2月24日—2011年10月5日），美国国家工程院院士，发明家、企业家，苹果公司联合创始人，曾任苹果公司首席执行官。乔布斯出生于加利福尼亚州 ... Jun 28, 2025 · 苹果公司（Apple）鲜为人知的第三位联合创始人罗纳德·韦恩（Ronald Wayne）最初被授予这家如今市值达3万亿美元的电脑公司10%的股份。 Sep 2, 2025 · 史蒂夫·乔布斯就是有“它”。从他创立苹果的那天到他去世的那天，他一直都有“它”。当然，他也有一些失误，谁知道他 ... Jul 3, 2025 · 在一些历史书籍的记载里，史蒂夫·乔布斯（Steve Jobs）和史蒂夫·沃兹尼亚克（Steve Wozniak）这两位大学辍学生摇身一变成为天才，于1976年创立了苹果电脑公司（Apple Computer ...'

In [3]:
# 溯源

from langchain_community.tools import DuckDuckGoSearchResults

search = DuckDuckGoSearchResults(output_format="list")

search.invoke("苹果公司的创始人 ?")

[{'snippet': 'May 5, 2025 · 史蒂芬·保罗·乔布斯（英语：Steven Paul Jobs，1955年2月24日－2011年10月5日），通称史蒂夫·乔布斯（英语：Steve Jobs），是一名美国企业家、营销家搭发明家，苹果公司个联合创始人之 ...',
  'title': '史蒂夫·乔布斯- 维基百科',
  'link': 'https://wuu.wikipedia.org/wiki/史蒂夫·乔布斯'},
 {'snippet': 'Dec 17, 2025 · 史蒂夫·乔布斯（Steve Jobs，1955年2月24日—2011年10月5日），美国国家工程院院士，发明家、企业家，苹果公司联合创始人，曾任苹果公司首席执行官。乔布斯出生于加利福尼亚州 ...',
  'title': '史蒂夫·乔布斯_百度百科',
  'link': 'https://baike.baidu.com/item/史蒂夫·乔布斯/85300'},
 {'snippet': "Mar 14, 2025 · 史蒂夫·沃茲尼亞克（Steve Wozniak），以及史蒂夫·喬布斯，發明了Apple I電腦。 蘋果電腦 成立於1976年。 當時有三個創始成員，兩個是史蒂夫（Steve's）和羅納德·韋恩（Ronald ...",
  'title': '史蒂夫•沃茲尼亞克（Steve Wozniak）',
  'link': 'https://pspeakers.com/zh-TW/揚聲器/史蒂夫·沃茲尼亞克/'},
 {'snippet': 'Jun 28, 2025 · 苹果公司（Apple）鲜为人知的第三位联合创始人罗纳德·韦恩（Ronald Wayne）最初被授予这家如今市值达3万亿美元的电脑公司10%的股份。',
  'title': '苹果联合创始人49年前以800美元清空股份，如今价值几何？ - 财富中文网',
  'link': 'https://www.fortunechina.com/shangye/c/2025-06/28/content_466815.htm'}]

In [1]:
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults
from langchain.agents import create_react_agent, AgentExecutor
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
# 创建搜索工具
search_wrapper = DuckDuckGoSearchResults(output_format="list")

@tool("my_search_tool")
def search(query: str) -> list[str]:
    """通过搜索引擎查询"""
    result = search_wrapper.invoke(query)
    return [res["snippet"] for res in result]

print(search.name)
print(search.description)
print(search.args)


def create_react_search_agent():
    tools = [search]
    # llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")
    llm = ChatOpenAI(
        temperature=0,
        api_key=os.getenv("DASHSCOPE_API_KEY"),
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
        model="qwen-turbo"
    )  
    
    
    prompt = PromptTemplate.from_template("""
        Answer the following questions as best you can. You have access to the following tools:

        {tools}

        Use the following format:

        Question: the input question you must answer
        Thought: you should always think about what to do
        Action: the action to take, should be one of [{tool_names}]
        Action Input: the input to the action
        Observation: the result of the action
        ... (this Thought/Action/Action Input/Observation can repeat N times)
        Thought: I now know the final answer
        Final Answer: the final answer to the original input question

        Begin!

        Question: {input}
        Thought:{agent_scratchpad}""")
    
    agent = create_react_agent(llm, tools, prompt)
    
    agent_executor = AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=True,
        max_iterations=3,
        handle_parsing_errors=True  # 这个很重要！
    )
    
    return agent_executor

# 使用修复后的 Agent
agent = create_react_search_agent()

# 测试
questions = ["苹果公司的创始人是谁？"]

for question in questions:
    print(f"\n问题: {question}")
    response = agent.invoke({"input": question})
    print(f"答案: {response['output']}")
    print("-" * 50)

/Users/anthony/miniforge3/envs/new_ai_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


my_search_tool
通过搜索引擎查询
{'query': {'title': 'Query', 'type': 'string'}}

问题: 苹果公司的创始人是谁？
苹果公司的创始人是史蒂夫·乔布斯（Steve Jobs）、史蒂夫·沃兹尼亚克（Steve Wozniak）和罗恩·韦恩（Ron Wayne）。其中，史蒂夫·乔布斯是苹果公司的主要创始人和领导者。

Final Answer: 苹果公司的创始人是史蒂夫·乔布斯（Steve Jobs）、史蒂夫·沃兹尼亚克（Steve Wozniak）和罗恩·韦恩（Ron Wayne）。

> Finished chain.
答案: 苹果公司的创始人是史蒂夫·乔布斯（Steve Jobs）、史蒂夫·沃兹尼亚克（Steve Wozniak）和罗恩·韦恩（Ron Wayne）。
--------------------------------------------------
